# 청년 인구 순유출 분석 — STEP 1: 전국 시군구 청년 순유출 현황

## 개요
KOSIS(국가통계포털)의 시군구별·성별·연령별 순이동 통계를 활용해, 전국 시군구 단위에서
청년층(19-34세)의 순유출 현황을 분석한다.

- 데이터 출처: KOSIS 국내인구이동통계 (시군구/성/연령별 순이동)
- 분석 기간: 2015~2025
- 원자료 형식: wide format (연도가 컬럼), cp949 인코딩


## 1. 데이터 불러오기

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

df = pd.read_csv(
    "../data/raw/kosis_sgg_sex_age_net_migration_raw.csv",
    encoding="cp949"
)
year_cols = [col for col in df.columns if "년" in col]
for c in year_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

print("원본 행 수:", len(df))
df.head()

원본 행 수: 10298


,행정구역(시군구)별,성별,연령별,항목,단위,2015 년,2016 년,2017 년,2018 년,2019 년,2020 년,2021 년,2022 년,2023 년,2024 년,2025 년,Unnamed: 16
0,서울특별시,남자,계,순이동[명],명,-74154.0,-74941.0,-55418.0,-62557.0,-33562.0,-41379.0,-58722.0,-22207.0,-20920,-26467.0,-18077.0,NaN
1,서울특별시,남자,0 - 4세,순이동[명],명,-8122.0,-8615.0,-6960.0,-7929.0,-4995.0,-4627.0,-4745.0,-3570.0,-3393,-3352.0,-2577.0,NaN
2,서울특별시,남자,5 - 9세,순이동[명],명,-3520.0,-3862.0,-2765.0,-3787.0,-1626.0,-2263.0,-2890.0,-1462.0,-721,-522.0,335.0,NaN
3,서울특별시,남자,10 - 14세,순이동[명],명,-1369.0,-1721.0,-896.0,-1338.0,-346.0,-812.0,-1310.0,-204.0,-35,105.0,316.0,NaN
4,서울특별시,남자,15 - 19세,순이동[명],명,-1147.0,-1294.0,-171.0,-312.0,653.0,-101.0,-689.0,1762.0,1624,1945.0,2182.0,NaN


## 2. 데이터 품질 진단 및 정제

원본 데이터를 그대로 사용할 경우 세 가지 문제가 있다.

1. **유령 행**: 마산시·진해시(2010년 창원시 통합), 여천시·여천군(1998년 여수시 통합) 등
   2015년 이전에 이미 폐지·통합된 옛 행정구역 코드가 남아 있으며, 분석 기간 전체 값이 0/결측이다.
2. **행정구역 관할 변경**: 군위군은 2023년 7월 1일 경상북도에서 대구광역시로 편입되어,
   데이터가 2015-2022(경북 관할)와 2023-2025(대구 관할)로 분리되어 있다.
3. **구/군 이름 중복**: 원본 표에 상위 시/도 컬럼이 없고, "중구", "동구", "남구" 등
   여러 광역시에서 반복되는 구 이름이 행 순서로만 구분되어 있다. 이름만으로 집계하면
   서로 다른 도시의 동명 구가 합산되는 오류가 발생한다 (예: 서울 중구 + 울산 중구가
   하나의 "중구"로 잘못 합산됨).

아래에서 상위 행정구역(parent) 컬럼을 복원하고, 유령 행을 제거하며,
군위군의 분리된 시계열을 2015 ~ 2022년까지는 경상북도, 2023 ~ 2025년도 까지의 데이터는 대구광역시로 설정한다.

In [2]:
parents = ['서울특별시','부산광역시','대구광역시','인천광역시','광주광역시','대전광역시',
           '울산광역시','세종특별자치시','경기도','강원도','강원특별자치도','충청북도','충청남도',
           '전라북도','전북특별자치도','전라남도','경상북도','경상남도','제주특별자치도']

# 시/도 행이 나올 때마다 갱신, 그 사이 구/군 행에는 forward fill로 소속 부여
df["parent"] = df["행정구역(시군구)별"].where(df["행정구역(시군구)별"].isin(parents)).ffill()
df["is_parent_row"] = df["행정구역(시군구)별"].isin(parents)

df[["parent", "행정구역(시군구)별", "is_parent_row"]].head(30)

,parent,행정구역(시군구)별,is_parent_row
0,서울특별시,서울특별시,True
1,서울특별시,서울특별시,True
2,서울특별시,서울특별시,True
3,서울특별시,서울특별시,True
4,서울특별시,서울특별시,True
5,서울특별시,서울특별시,True
6,서울특별시,서울특별시,True
7,서울특별시,서울특별시,True
8,서울특별시,서울특별시,True
9,서울특별시,서울특별시,True


# 군위군 행정구역 변경 처리
- 군위군은 2023년 7월 1일 경상북도에서 대구광역시로 편입됨
- 경상북도 군위군의 2023년 값은 0으로 기록되어 있으므로 관할 변경에 따른 플레이스홀더로 보고 결측 처리
- 대구광역시 군위군의 2023년 실제 값은 그대로 유지

In [3]:
gunwi_gyeongbuk = (
    (df["parent"] == "경상북도") &
    (df["행정구역(시군구)별"] == "군위군")
)

df.loc[gunwi_gyeongbuk, "2023 년"] = np.nan

In [4]:
gunwi_check = df[
    (df["행정구역(시군구)별"] == "군위군") &
    (df["parent"].isin(["경상북도", "대구광역시"]))
].copy()

# 군위군 청년층 데이터만 확인

gunwi_youth_check = gunwi_check[
    gunwi_check["연령별"] == "청년층(19-34세)"
].copy()

gunwi_youth_check[
    [
        "parent",
        "행정구역(시군구)별",
        "성별",
        "연령별",
        "2022 년",
        "2023 년",
        "2024 년",
        "2025 년"
    ]
].sort_values(["parent", "성별"])

,parent,행정구역(시군구)별,성별,연령별,2022 년,2023 년,2024 년,2025 년
8682,경상북도,군위군,남자,청년층(19-34세),-21.0,NaN,NaN,NaN
8701,경상북도,군위군,여자,청년층(19-34세),26.0,NaN,NaN,NaN
1994,대구광역시,군위군,남자,청년층(19-34세),NaN,-67.0,-123.0,-16.0
2013,대구광역시,군위군,여자,청년층(19-34세),NaN,-118.0,-66.0,-6.0


### 인천 남구 → 미추홀구 개명 병합

인천 남구는 2018년 7월 1일 미추홀구로 개칭되었다(행정구역 관할 변경이 아닌 단순 개명).
두 이름 모두 전환 연도(2018)에 0이 아닌 값을 가지고 있는데, 이는 개명 시점(7월 1일) 기준으로
연도 중간에 코드가 바뀌면서 상반기는 남구, 하반기는 미추홀구로 나뉘어 보고된 것으로 판단된다.
따라서 군위군과 달리 경계 연도를 결측 처리하지 않고, 남구의 지역명을 미추홀구로 통일한 뒤
합산하여 하나의 연속된 시계열로 병합한다.

In [ ]:
# 인천 남구는 원본에 '남  구'(공백 2칸)로 저장되어 있음 — repr()로 확인함
mask_namgu = (df["parent"] == "인천광역시") & (df["행정구역(시군구)별"] == "남  구")
df.loc[mask_namgu, "행정구역(시군구)별"] = "미추홀구"

# 같은 이름이 된 행들(원래 남구였던 행 + 원래 미추홀구였던 행)을 합산
mask_michuhol = (df["parent"] == "인천광역시") & (df["행정구역(시군구)별"] == "미추홀구")

michuhol_merged = (
    df[mask_michuhol]
    .groupby(["parent", "행정구역(시군구)별", "성별", "연령별"], as_index=False)[year_cols]
    .sum(min_count=1)
)
michuhol_merged["is_parent_row"] = False

df = pd.concat([df[~mask_michuhol], michuhol_merged], ignore_index=True)
print("병합 후 행 수:", len(df))

# 검증: 미추홀구 청년층 시계열이 연속적으로 나오는지 확인
check = df[
    (df["parent"] == "인천광역시") &
    (df["행정구역(시군구)별"] == "미추홀구") &
    (df["연령별"] == "청년층(19-34세)")
]
check[["성별"] + year_cols]

병합 후 행 수: 10260


,성별,2015 년,2016 년,2017 년,2018 년,2019 년,2020 년,2021 년,2022 년,2023 년,2024 년,2025 년
10240,남자,-501.0,1793.0,1027.0,281.0,-575.0,111.0,1386.0,216.0,58.0,1235.0,971.0
10259,여자,-638.0,1790.0,435.0,-7.0,-752.0,194.0,1007.0,134.0,-220.0,930.0,857.0


In [ ]:
# 전 기간(2015~2025) 완전 0/결측인 유령 행 탐지

ghost_regions = (
    df.groupby(["parent", "행정구역(시군구)별"])[year_cols]
    .apply(lambda g: (g.fillna(0) == 0).all().all())
)

ghost_keys = set(ghost_regions[ghost_regions].index)
print(f"유령 행 제거 대상: {len(ghost_keys)}개")
print(ghost_keys)

df["_key"] = list(zip(df["parent"], df["행정구역(시군구)별"]))
df = df[~df["_key"].isin(ghost_keys)].drop(columns="_key").copy()
print("제거 후 행 수:", len(df))

유령 행 제거 대상: 22개
{('제주특별자치도', '북제주군'), ('충청남도', '논산군'), ('경상남도', '마산시'), ('경기도', '양주군'), ('경기도', '광주군'), ('경기도', '파주군'), ('전라남도', '여천시'), ('충청남도', '당진군'), ('제주특별자치도', '국외 및 기타'), ('경기도', '시흥군'), ('경상남도', '진해시'), ('경기도', '김포군'), ('전라남도', '여천군'), ('충청남도', '연기군'), ('경기도', '화성군'), ('경기도', '포천군'), ('경상남도', '양산군'), ('경기도', '안성군'), ('경기도', '용인군'), ('경기도', '이천군'), ('제주특별자치도', '남제주군'), ('경상남도', '울산시')}
제거 후 행 수: 9424


In [7]:
# 검증: parent+region 조합 기준 고유 시군구 수, 이름만으로 셌을 때와 비교
combo = df[~df["is_parent_row"]][["parent", "행정구역(시군구)별"]].drop_duplicates()
print("이름 기준 고유 지역 수:", df[~df["is_parent_row"]]["행정구역(시군구)별"].nunique())
print("parent+이름 기준 고유 지역 수:", len(combo))

# 동명 구 중복 사례 확인 (예: 중구)
combo[combo["행정구역(시군구)별"] == "중구"]

이름 기준 고유 지역 수: 211
parent+이름 기준 고유 지역 수: 229


,parent,행정구역(시군구)별
76,서울특별시,중구
2888,울산광역시,중구


## 3. 전처리: long format 변환 및 청년층 추출

시/도 자체 합계 행(`is_parent_row=True`)은 시군구 단위 분석에서 제외한다. 이 행들을 포함하면
"서울특별시"(광역 합계)와 "강남구"(구 단위)가 같은 랭킹에서 비교되는 단위 불일치 문제가 생긴다.

In [8]:
# 시도 단위 행 제거
# 단, 세종특별자치시는 시군구 단위 분석에 포함
df_sgg = df[
    (~df["is_parent_row"]) |
    (df["parent"] == "세종특별자치시")
].copy()

df_long = df_sgg.melt(
    id_vars=["parent", "행정구역(시군구)별", "성별", "연령별"],
    value_vars=year_cols,
    var_name="year",
    value_name="net_migration"
)
df_long = df_long.rename(columns={
    "행정구역(시군구)별": "region",
    "성별": "sex",
    "연령별": "age_group"
})
df_long["year"] = df_long["year"].str.replace(" 년", "", regex=False).astype(int)
df_long.head()

,parent,region,sex,age_group,year,net_migration
0,서울특별시,종로구,남자,계,2015,-1276.0
1,서울특별시,종로구,남자,0 - 4세,2015,-50.0
2,서울특별시,종로구,남자,5 - 9세,2015,-4.0
3,서울특별시,종로구,남자,10 - 14세,2015,-15.0
4,서울특별시,종로구,남자,15 - 19세,2015,44.0


In [9]:
df_youth = df_long[df_long["age_group"] == "청년층(19-34세)"].copy()
df_youth = df_youth.dropna(subset=["net_migration"]).copy()

# 2023년 행정구역 변경에 따른 경북 군위군 제거
df_youth = df_youth[
    ~(
        (df_youth["parent"] == "경상북도") &
        (df_youth["region"] == "군위군") &
        (df_youth["year"] == 2023)
    )
].copy()

print(df_youth.shape)
df_youth.isna().sum()

(5082, 6)


parent           0
region           0
sex              0
age_group        0
year             0
net_migration    0
dtype: int64

성별 데이터를 합산해 지역 단위 순이동으로 집계한다. 지역은 `parent`(상위 시/도)와
`region`(시군구명)을 함께 사용해 동명 구 문제를 피한다.
(성별에 따른 이동 패턴 차이는 이후 STEP3 이동 사유 분석에서 별도로 다룬다)

In [10]:
# 성별 데이터를 합산해 지역 단위 순이동 계산
df_youth_total = (
    df_youth
    .groupby(
        ["year", "parent", "region"],
        as_index=False
    )["net_migration"]
    .sum()
)

# 지역명 내부의 불필요한 공백 제거
# 예: "남  구" → "남구", "동  구" → "동구"
df_youth_total["region"] = (
    df_youth_total["region"]
    .str.replace(r"\s+", "", regex=True)
)

# 시도 + 시군구를 결합해 고유 지역명 생성
# 세종특별자치시는 시군구명이 동일하므로 중복 표기를 방지
df_youth_total["region_full"] = np.where(
    df_youth_total["parent"] == "세종특별자치시",
    "세종특별자치시",
    df_youth_total["parent"] + " " + df_youth_total["region"]
)

df_youth_total.head()

,year,parent,region,net_migration,region_full
0,2015,강원특별자치도,강릉시,-1378.0,강원특별자치도 강릉시
1,2015,강원특별자치도,고성군,-367.0,강원특별자치도 고성군
2,2015,강원특별자치도,동해시,-585.0,강원특별자치도 동해시
3,2015,강원특별자치도,삼척시,-1517.0,강원특별자치도 삼척시
4,2015,강원특별자치도,속초시,-296.0,강원특별자치도 속초시


In [11]:
# 1. 시도 개수
print("시도 개수:", df_youth_total["parent"].nunique())

# 2. 시도 목록
print("\n시도 목록:")
print(sorted(df_youth_total["parent"].unique()))

# 3. 시도별 시군구 개수
print("\n시도별 시군구 개수:")
print(
    df_youth_total
    .groupby("parent")["region_full"]
    .nunique()
)

# 4. 전체 시군구 개수
print("\n전체 시군구 개수:", df_youth_total["region_full"].nunique())

# 5. 연도별 시군구 개수
print("\n연도별 시군구 개수:")
print(
    df_youth_total
    .groupby("year")["region_full"]
    .nunique()
)

시도 개수: 17

시도 목록:
['강원특별자치도', '경기도', '경상남도', '경상북도', '광주광역시', '대구광역시', '대전광역시', '부산광역시', '서울특별시', '세종특별자치시', '울산광역시', '인천광역시', '전라남도', '전북특별자치도', '제주특별자치도', '충청남도', '충청북도']

시도별 시군구 개수:
parent
강원특별자치도    18
경기도        31
경상남도       18
경상북도       23
광주광역시       5
대구광역시       9
대전광역시       5
부산광역시      16
서울특별시      25
세종특별자치시     1
울산광역시       5
인천광역시      10
전라남도       22
전북특별자치도    14
제주특별자치도     2
충청남도       15
충청북도       11
Name: region_full, dtype: int64

전체 시군구 개수: 230

연도별 시군구 개수:
year
2015    229
2016    229
2017    229
2018    229
2019    229
2020    229
2021    229
2022    229
2023    229
2024    229
2025    229
Name: region_full, dtype: int64


## 4. 전처리 결과 저장

이후 단계에서 재사용할 수 있도록 전처리된 데이터를 저장한다.

In [12]:
df_youth_total.to_csv(
    "../data/processed/youth_net_migration_processed.csv",
    index=False,
    encoding="utf-8-sig"
)
print("저장 완료, 행 수:", len(df_youth_total))

저장 완료, 행 수: 2519


## 5. 분석 시작